# Create SQL Views in DBRepo

Creates the three views defined in [`../data/views.sql`](../data/views.sql) in the SDCC Traffic Flow Experiment 2022 database on DBRepo.

All three views are denormalising joins of `TrafficMeasurements` with `Calendar` and `TrafficSites`, two of them with simple WHERE filters. They are expressible in DBRepo's structured `QueryDefinition` model — feature engineering (cyclical time, target binning, class balancing, hourly aggregation) is done downstream in Python.

Prerequisite: the schema and data must already be loaded — see [`upload-data-DBrepo-notebook.ipynb`](upload-data-DBrepo-notebook.ipynb).

DBRepo API documentation: https://www.ifs.tuwien.ac.at/infrastructures/dbrepo/dbrepo_v1.13.3.pdf

## 1. Set up the REST client

In [ ]:
import os
import time
from getpass import getpass
from dotenv import load_dotenv

from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    QueryDefinition,
    JoinDefinition,
    JoinType,
    ConditionalDefinition,
    FilterDefinition,
    FilterType,
)

# Load shared config from ../.env (DBREPO_ENDPOINT, DBREPO_USERNAME,
# DBREPO_DATABASE_ID, etc.). See .env.example in the repo root.
load_dotenv("../.env")
DATABASE_ID = os.environ["DBREPO_DATABASE_ID"]
print("DATABASE_ID:", DATABASE_ID)

In [ ]:
# --- Compatibility patch for dbrepo 1.13.5 query_to_subset() ---
# Two bugs in the SDK's mapper.query_to_subset() block valid view
# creation in this project:
#
#   1. It calls `np.array([list_of_lists]).flatten()` on lists with
#      different inner lengths (one table's columns vs another).
#      numpy >= 1.24 refuses to build implicit ragged arrays.
#
#   2. The select_columns block filters columns by membership in the
#      query.columns suffix list, without checking that the matched
#      column belongs to the table the query asked for. So when two
#      tables have a column with the same name (e.g. Calendar.date and
#      TrafficMeasurements.date), referencing one matches both and the
#      count check raises "found > expected".
#
# This patch replaces query_to_subset entirely with a per-query-column
# lookup that resolves each "table.column" string against the named
# table specifically. Other behaviour (joins, filters, orders, error
# messages) is preserved.

import dbrepo.api.mapper as _mapper
import dbrepo.RestClient as _restclient
from dbrepo.api.dto import (
    Subset, SubsetColumn, Filter, FilterType, Join, Conditional, Order,
)
from dbrepo.api.exceptions import MalformedError


def _patched_query_to_subset(database, image, query):
    # --- validation ---
    if len(query.columns) < 1:
        raise MalformedError("Failed to map subset: no column(s) selected")
    if len(query.datasources) < 1:
        raise MalformedError("Failed to map subset: no datasource(s) selected")

    wrong = [c for c in query.columns if len(c.split(".")) != 2]
    for join in (query.joins or []):
        for cond in join.conditionals:
            if len(cond.column.split(".")) != 2 or len(cond.foreign_column.split(".")) != 2:
                wrong.append(cond)
    for filt in (query.filters or []):
        if filt.type == FilterType.WHERE and len(filt.column.split(".")) != 2:
            wrong.append(filt.column)
    for order in (query.orders or []):
        if len(order.column.split(".")) != 2:
            wrong.append(order.column)
    if wrong:
        raise MalformedError(
            f"Failed to map subset: column(s) are not in table.column notion: {wrong}"
        )

    # --- lookup helpers ---
    sources_by_name = {t.internal_name: t for t in database.tables}
    for v in (database.views or []):
        sources_by_name.setdefault(v.internal_name, v)

    def _find_source(name):
        return sources_by_name.get(name)

    def _find_column(qualified):
        table_name, col_name = qualified.split(".")
        src = _find_source(table_name)
        if src is None:
            return None
        return next((c for c in src.columns if c.internal_name == col_name), None)

    # --- columns ---
    select_columns = []
    for q in query.columns:
        col = _find_column(q)
        if col is None:
            raise MalformedError(
                f"Failed to map subset: column {q} not found in database"
            )
        select_columns.append(SubsetColumn(id=col.id))

    # --- datasources ---
    datasource_ids = []
    for ds in query.datasources:
        src = _find_source(ds)
        if src is None:
            raise MalformedError(
                f"Failed to map subset: datasource {ds} not found in database"
            )
        datasource_ids.append(src.id)

    # --- joins ---
    joins = []
    for join in (query.joins or []):
        join_src = _find_source(join.datasource)
        if join_src is None:
            raise MalformedError(
                f"Failed to map subset: join datasource {join.datasource} not found in database"
            )
        conditionals = []
        for cond in join.conditionals:
            col = _find_column(cond.column)
            if col is None:
                raise MalformedError(
                    f"Failed to map subset: conditional column {cond.column} not found in database"
                )
            fcol = _find_column(cond.foreign_column)
            if fcol is None:
                raise MalformedError(
                    f"Failed to map subset: conditional column {cond.foreign_column} not found in database"
                )
            conditionals.append(Conditional(column_id=col.id, foreign_column_id=fcol.id))
        joins.append(Join(type=join.type, datasource_id=join_src.id, conditionals=conditionals))

    # --- filters ---
    filters = []
    for filt in (query.filters or []):
        if filt.type != FilterType.WHERE:
            filters.append(Filter(type=filt.type))
            continue
        col = _find_column(filt.column)
        if col is None:
            raise MalformedError(
                f"Failed to map subset: filtered column {filt.column} not found in database: {database.internal_name}"
            )
        op_ids = [op.id for op in image.operators if op.value == filt.operator]
        if len(op_ids) != 1:
            raise MalformedError(
                f"Failed to map subset: filter operator {filt.operator} not found in image"
            )
        filters.append(Filter(
            type=filt.type,
            column_id=col.id,
            operator_id=op_ids[0],
            value=filt.value,
        ))

    # --- orders ---
    orders = []
    for order in (query.orders or []):
        col = _find_column(order.column)
        if col is None:
            raise MalformedError(
                f"Failed to map subset: order column {order.column} not found in database: {database.internal_name}"
            )
        orders.append(Order(column_id=col.id, direction=order.direction))

    return Subset(
        columns=select_columns,
        datasource_ids=datasource_ids,
        joins=joins,
        filters=filters,
        orders=orders,
    )


# Patch both bindings: the source module and the import in RestClient.
_mapper.query_to_subset = _patched_query_to_subset
_restclient.query_to_subset = _patched_query_to_subset

print("query_to_subset patched.")

In [ ]:
DBREPO_ENDPOINT = os.environ["DBREPO_ENDPOINT"]
DATABASE_ID     = os.environ["DBREPO_DATABASE_ID"]
username        = os.environ["DBREPO_USERNAME"]

if not DATABASE_ID:
    raise RuntimeError(
        "DBREPO_DATABASE_ID is not set in .env — run the upload notebook "
        "first to create the database, then paste its id into .env."
    )

user_password = getpass(f"Password for {username}: ")

client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=username,
    password=user_password,
)
print("Current user:", client.whoami())

## Optional: inspect visible databases

This helps students understand what databases they can currently see.


In [10]:
databases = client.get_databases()
print(f"Visible databases: {len(databases)}")

for db in databases[:10]:
    print(f"- {db.id} | {db.name}")


Visible databases: 14
- 2de50b61-ac24-4484-8117-dc0fe9dc1b7c | SDCC Traffic Flow Experiment 2022
- f168077a-72ae-449e-a485-be4aae0d5490 | SDCC Traffic Flow Experiment 2022
- 3394d25b-b0ee-4a91-bd4a-f5d5d34cc68c | SDCC Traffic Flow Experiment 2022
- 0fb6d4a0-faea-4f42-8bfc-75e97fc2e710 | SDCC Traffic Flow Experiment 2022
- be8863ae-9794-4ee2-a964-3e22133b3840 | Data_Stewardship_Group14_Football_player_prediction
- a16a980a-3489-492b-adcf-74c70a07248b | frost_day_prediction_in_vienna
- 899bfcba-7fec-40c9-9076-3a3a9372c844 | vienna_weather_wet_months
- 412fb0ce-5299-4d0e-a271-4641b1365b8a | data_stewardship_g12_unemployment_prediction
- 123289f2-5218-4b32-b962-5f3dafec1fe3 | ev_protection_investment_analysis
- 38707917-e942-45c3-a3dd-d2bfc1c106af | Vienna Demographic Forecasting


## 2. Discover the tables' internal names

DBRepo generates a normalised `internal_name` for each table when it is created (typically lowercase, with non-ASCII replaced). `QueryDefinition` references tables and columns by these internal names in `"table.column"` form, so we look them up here once and reuse below.

In [11]:
database = client.get_database(database_id=DATABASE_ID)
tables_by_display_name = {t.name: t.internal_name for t in database.tables}
print(tables_by_display_name)

T_MEAS = tables_by_display_name["TrafficMeasurements"]
T_CAL  = tables_by_display_name["Calendar"]
T_SITE = tables_by_display_name["TrafficSites"]

{'TrafficMeasurements': 'trafficmeasurements', 'Calendar': 'calendar', 'TrafficSites': 'trafficsites'}


## 3. Create the views

Each view is built from a `QueryDefinition`. The Python SDK converts the structured definition into the database-level SQL view DBRepo registers. If you need to re-run after editing a view, use section 4 to drop it first.

In [ ]:
# Common join shape reused by all three views.
# Note: the FK column in both tables is `date`, not `date_id` —
# this is how the upload notebook materialised the schema in DBRepo,
# even though data/create.sql documents it as `date_id`.
join_calendar = JoinDefinition(
    type=JoinType.INNER,
    datasource=T_CAL,
    conditionals=[
        ConditionalDefinition(
            column=f"{T_MEAS}.date",
            foreign_column=f"{T_CAL}.date",
        )
    ],
)

join_sites = JoinDefinition(
    type=JoinType.INNER,
    datasource=T_SITE,
    conditionals=[
        ConditionalDefinition(
            column=f"{T_MEAS}.site_id",
            foreign_column=f"{T_SITE}.site_id",
        )
    ],
)

# Columns the views expose, in display order.
view_columns = [
    f"{T_MEAS}.observation_id",
    f"{T_MEAS}.site_id",
    f"{T_MEAS}.date",
    f"{T_CAL}.day_of_week",
    f"{T_MEAS}.start_time",
    f"{T_MEAS}.end_time",
    f"{T_MEAS}.flow",
    f"{T_MEAS}.flow_pc",
    f"{T_MEAS}.cong",
    f"{T_MEAS}.cong_pc",
    f"{T_MEAS}.dsat",
    f"{T_MEAS}.dsat_pc",
]

In [ ]:
# 1. v_measurements_enriched -- full denormalised join, no filter.
client.create_view(
    database_id=DATABASE_ID,
    name="v_measurements_enriched",
    query=QueryDefinition(
        columns=view_columns,
        datasources=[T_MEAS],
        joins=[join_calendar, join_sites],
    ),
    is_public=True,
    is_schema_public=True,
)

In [ ]:
# 2. v_weekday_measurements -- filter day_of_week to Mon..Fri.
# Calendar.day_of_week stores 2-letter codes: MO/TU/WE/TH/FR/SA/SU.
# DBRepo only accepts a single filter clause (the bare AND/OR connector
# the SDK supports is rejected by the data-service), so use NOT IN with
# a comma-separated value list.
client.create_view(
    database_id=DATABASE_ID,
    name="v_weekday_measurements",
    query=QueryDefinition(
        columns=view_columns,
        datasources=[T_MEAS],
        joins=[join_calendar, join_sites],
        filters=[
            FilterDefinition(
                type=FilterType.WHERE,
                column=f"{T_CAL}.day_of_week",
                operator="NOT IN",
                value="SA,SU",
            ),
        ],
    ),
    is_public=True,
    is_schema_public=True,
)

In [ ]:
# 3. v_peak_hour_measurements -- start_time inside the morning (07:00-09:00)
# or evening (17:00-19:00) rush windows.
# Bounds are inclusive on both sides and start_time is sampled at 15-minute
# intervals, so enumerating the qualifying slots (9 + 9 = 18 values) lets
# us express the disjoint ranges as a single IN filter -- the only shape
# DBRepo's data-service currently accepts cleanly.
PEAK_QUARTERS = ",".join(
    f"{h:02d}:{m:02d}"
    for h in (7, 8, 17, 18)
    for m in (0, 15, 30, 45)
) + ",09:00,19:00"  # include the upper bounds explicitly

client.create_view(
    database_id=DATABASE_ID,
    name="v_peak_hour_measurements",
    query=QueryDefinition(
        columns=view_columns,
        datasources=[T_MEAS],
        joins=[join_calendar, join_sites],
        filters=[
            FilterDefinition(
                type=FilterType.WHERE,
                column=f"{T_MEAS}.start_time",
                operator="IN",
                value=PEAK_QUARTERS,
            ),
        ],
    ),
    is_public=True,
    is_schema_public=True,
)

In [ ]:
# Clean up the three diagnostic views (__diag_single, __diag_two_where,
# __diag_not_in) created while figuring out the right filter shape.
for v in client.get_views(database_id=DATABASE_ID):
    if v.name.startswith("__diag_"):
        client.delete_view(database_id=DATABASE_ID, view_id=v.id)
        print(f"Dropped diagnostic view {v.name}")

## 4. Verify

List the views the database now has, then pull a small sample from each.

In [ ]:
views = client.get_views(database_id=DATABASE_ID)
views_by_name = {v.name: v.id for v in views}
for name, vid in views_by_name.items():
    print(f"{name}: {vid}")

In [ ]:
# Views are mapped asynchronously after create_view returns. Poll
# get_view_data, retrying on the 204 No Content the server returns
# while the view is still being materialised. Pattern mirrors the
# DBRepo ingestion guide notebook.

def fetch_view_sample(view_id, size=5, retries=10, delay=5):
    for attempt in range(retries):
        try:
            return client.get_view_data(
                database_id=DATABASE_ID,
                view_id=view_id,
                page=0,
                size=size,
            )
        except Exception as e:
            if "204" in str(e):
                print(f"  view not ready yet, waiting {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"View {view_id} did not become ready after {retries * delay}s")


for name in [
    "v_measurements_enriched",
    "v_weekday_measurements",
    "v_peak_hour_measurements",
]:
    if name not in views_by_name:
        print(f"{name}: NOT FOUND")
        continue
    print(f"\n--- {name} ---")
    df = fetch_view_sample(views_by_name[name], size=5)
    n = client.get_view_data_count(
        database_id=DATABASE_ID, view_id=views_by_name[name]
    )
    print(f"row count: {n}")
    print(df.head())

## 5. Optional: drop views

Uncomment when you need to recreate a view (e.g. after editing it).

In [ ]:
# for name in [
#     "v_peak_hour_measurements",
#     "v_weekday_measurements",
#     "v_measurements_enriched",
# ]:
#     if name in views_by_name:
#         client.delete_view(database_id=DATABASE_ID, view_id=views_by_name[name])
#         print(f"dropped {name}")